In [ ]:
import os
import qdrant_client
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.storage.docstore import SimpleDocumentStore

# Setup
collectionname = "WAMASRAGHIERARCHICAL"
url_embedder = os.getenv("VLLM_API_BASE_URL")
url_qdrant = os.getenv("QDRANT_URL")
url_llm = os.getenv("VLLM_API_BASE_URL")

# Client Qdrant
client = qdrant_client.QdrantClient(url=url_qdrant)

# Embedding Model
embed_model = OpenAIEmbedding(
    api_base=url_embedder,
    model_name="BAAI/bge-m3",
    api_key="null",
)
Settings.embed_model = embed_model

# Qdrant
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collectionname,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)

# LLM
llm = OpenAILike(
    model="Qwen/Qwen2.5-32B-Instruct-AWQ",
    api_base=url_llm, 
    api_key="fake-key",                 
    is_chat_model=True,
    timeout=600.0,
    temperature=0.7,
    context_window=8192                       
)

# carica tutto dal disco
docstore = SimpleDocumentStore.from_persist_dir(persist_dir="../chunking/storage_hierarchical")

# oggetto storagecontext importante, lo passiamo a index
storage_context = StorageContext.from_defaults(
    vector_store=vector_store,
    docstore=docstore  
)

# index basato sulle foglie di qdrant
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    storage_context=storage_context
)

# troviamo top 10 foglie con hybrid search
base_retriever = index.as_retriever(
    similarity_top_k=10,  
    vector_store_query_mode="hybrid"  
)

# configurazione automergingretriever
retriever = AutoMergingRetriever(
    base_retriever,
    storage_context,  
    verbose=True,  
    simple_ratio_thresh=0.3  # 30% di figli = da il padre
)


query_engine = RetrieverQueryEngine.from_args(
    retriever,
    llm=llm
)

# --- Test ---
response = query_engine.query("quali scelte ci sono per le piastrelle?")
print(str(response))

> Merging 4 nodes into parent node.
> Parent node id: 4912b4cf-084a-46f3-bb0e-b43aa61c10a0.
> Parent node text: <!-- image -->

## CRITERI DI TONALIZZAZIONE E GESTIONE SOTTOSCELTE

Last revision date: 11.12.20...

> Merging 2 nodes into parent node.
> Parent node id: bdeec05e-bf95-42fa-9f1a-3313683ee8f9.
> Parent node text: ## Summary

|   1. | Introduzione cambio setup degli AGV............................................

> Merging 1 nodes into parent node.
> Parent node id: f84445c7-9c04-49b7-aa82-a301bb437c1b.
> Parent node text: ## Summary

|   1. | Introduzione cambio setup degli AGV............................................

Le scelte per le piastrelle sono A (PRIMA COMMERCIALE) e S (SOTTOSCELTA).
